In [1]:
!pip -q install -U anthropic datasets tqdm tenacity scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.4 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
print("Anthropic API key loaded securely.")

Enter your Anthropic API key: ··········
Anthropic API key loaded securely.


In [3]:
import os
import re
import json
import time
import pandas as pd

from json import JSONDecodeError
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset

from anthropic import (
    Anthropic,
    RateLimitError,
    APIConnectionError,
    BadRequestError,
    AuthenticationError
)

from tenacity import (
    retry,
    wait_exponential,
    stop_after_attempt,
    retry_if_exception_type
)

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

MODEL = "claude-opus-4-7"
DATASET_NAME = "ChanceFocus/flare-cd"

# First test with 100 samples.
# Set LIMIT = None for the full test set.
LIMIT = 100

dataset = load_dataset(DATASET_NAME)

split_name = "test" if "test" in dataset else list(dataset.keys())[0]
split_data = dataset[split_name]

print(dataset)
print("Split:", split_name)
print("Columns:", split_data.column_names)
print("Example:", split_data[0])

VALID_LABELS = [
    "O",
    "B-CAUSE", "I-CAUSE",
    "B-EFFECT", "I-EFFECT"
]

SYSTEM_PROMPT = """
You are a cause-effect sequence labeling model.

Task:
Given a list of tokens from a text section, assign exactly one BIO label to each token.

Allowed labels:
- O
- B-CAUSE, I-CAUSE
- B-EFFECT, I-EFFECT

Definitions:
- CAUSE: the text span that represents the cause of an event.
- EFFECT: the text span that represents the effect or result caused by the cause.
- O: tokens that are not part of either a cause or effect span.

Rules:
1. Return exactly one label for each input token.
2. The number of output labels must equal the number of input tokens.
3. Use only the allowed labels.
4. Do not add explanations.
5. Return only valid JSON in this exact format:
{"labels": ["B-EFFECT", "I-EFFECT", "O", "B-CAUSE"]}
"""

def get_column_name(example, possible_names):
    for name in possible_names:
        if name in example:
            return name

    raise ValueError(
        f"Cannot find any column from {possible_names}. "
        f"Available columns are: {list(example.keys())}"
    )

def normalize_label(label):
    label = str(label).strip()

    if label in VALID_LABELS:
        return label

    return "O"

def parse_claude_json(raw_text):
    raw_text = str(raw_text).strip()

    if raw_text == "":
        raise ValueError("Empty model output")

    try:
        return json.loads(raw_text)

    except JSONDecodeError:
        match = re.search(r"\{.*\}", raw_text, flags=re.DOTALL)

        if match:
            return json.loads(match.group(0))

        print("Raw model output:")
        print(repr(raw_text))
        raise

def fix_prediction_length(pred_labels, target_len):
    pred_labels = [normalize_label(x) for x in pred_labels]

    if len(pred_labels) < target_len:
        pred_labels = pred_labels + ["O"] * (target_len - len(pred_labels))

    if len(pred_labels) > target_len:
        pred_labels = pred_labels[:target_len]

    return pred_labels

def bio_to_spans(labels):
    spans = []
    start = None
    span_type = None

    for i, label in enumerate(labels + ["O"]):
        if label == "O":
            if start is not None:
                spans.append((start, i, span_type))
                start = None
                span_type = None
            continue

        if "-" not in label:
            continue

        prefix, label_type = label.split("-", 1)

        if prefix == "B":
            if start is not None:
                spans.append((start, i, span_type))
            start = i
            span_type = label_type

        elif prefix == "I":
            if start is None:
                start = i
                span_type = label_type
            elif span_type != label_type:
                spans.append((start, i, span_type))
                start = i
                span_type = label_type

    return spans

def compute_metrics(gold_all, pred_all):
    correct_tokens = 0
    total_tokens = 0
    exact_match_count = 0

    tp = 0
    fp = 0
    fn = 0

    for gold_labels, pred_labels in zip(gold_all, pred_all):
        total_tokens += len(gold_labels)
        correct_tokens += sum(g == p for g, p in zip(gold_labels, pred_labels))

        if gold_labels == pred_labels:
            exact_match_count += 1

        gold_spans = Counter(bio_to_spans(gold_labels))
        pred_spans = Counter(bio_to_spans(pred_labels))

        tp += sum((gold_spans & pred_spans).values())
        fp += sum((pred_spans - gold_spans).values())
        fn += sum((gold_spans - pred_spans).values())

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    token_accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0.0
    exact_match_accuracy = exact_match_count / len(gold_all) if len(gold_all) > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "token_accuracy": token_accuracy,
        "exact_match_accuracy": exact_match_accuracy,
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn
    }

@retry(
    retry=retry_if_exception_type((RateLimitError, APIConnectionError, JSONDecodeError, ValueError)),
    wait=wait_exponential(multiplier=5, min=10, max=120),
    stop=stop_after_attempt(5),
    reraise=True
)
def call_claude_cd(tokens):
    indexed_tokens = "\n".join([f"{i}: {tok}" for i, tok in enumerate(tokens)])

    user_prompt = (
        f"Number of tokens: {len(tokens)}\n\n"
        f"Tokens:\n{indexed_tokens}\n\n"
        f"Return exactly {len(tokens)} BIO labels as JSON."
    )

    try:
        message = client.messages.create(
            model=MODEL,
            max_tokens=1500,
            system=SYSTEM_PROMPT,
            messages=[
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
        )

    except BadRequestError as e:
        print("\nBadRequestError detail:")
        print(e)
        raise

    except AuthenticationError as e:
        print("\nAuthenticationError detail:")
        print(e)
        raise

    raw_output = "".join(
        block.text for block in message.content
        if getattr(block, "type", None) == "text"
    )

    parsed = parse_claude_json(raw_output)

    if "labels" not in parsed:
        raise ValueError(f"Missing labels field in model output: {parsed}")

    return parsed

example = split_data[0]

token_col = get_column_name(example, ["token", "tokens"])
label_col = get_column_name(example, ["label", "labels", "tags"])

print("Token column:", token_col)
print("Label column:", label_col)

sample_size = len(split_data) if LIMIT is None else min(LIMIT, len(split_data))
eval_data = split_data.select(range(sample_size))

rows = []
gold_all = []
pred_all = []

for idx, row in enumerate(tqdm(eval_data)):
    tokens = row[token_col]
    gold_labels = [normalize_label(x) for x in row[label_col]]

    try:
        result = call_claude_cd(tokens)
        pred_labels = result.get("labels", [])
        pred_labels = fix_prediction_length(pred_labels, len(gold_labels))
        error = ""

    except BadRequestError as e:
        print(f"\nStopping at row {idx} because of BadRequestError.")
        print(e)
        raise

    except AuthenticationError as e:
        print(f"\nStopping at row {idx} because of AuthenticationError.")
        print(e)
        raise

    except Exception as e:
        pred_labels = ["O"] * len(gold_labels)
        error = str(e)
        print(f"Error at row {idx}: {e}")

    gold_all.append(gold_labels)
    pred_all.append(pred_labels)

    rows.append({
        "idx": idx,
        "id": row.get("id", ""),
        "text": row.get("text", ""),
        "tokens": tokens,
        "gold_labels": gold_labels,
        "pred_labels": pred_labels,
        "gold_spans": bio_to_spans(gold_labels),
        "pred_spans": bio_to_spans(pred_labels),
        "exact_match": gold_labels == pred_labels,
        "error": error
    })

    # Slow down to reduce rate-limit risk.
    time.sleep(5)

metrics = compute_metrics(gold_all, pred_all)

print("\n===== Claude Opus 4.7 FLARE-CD Evaluation =====")
print(f"Dataset: {DATASET_NAME}")
print(f"Split: {split_name}")
print(f"Model: {MODEL}")
print(f"Samples evaluated: {len(eval_data)}")
print(f"Precision / Correctness Rate: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1_score']:.4f}")
print(f"Exact Match Accuracy: {metrics['exact_match_accuracy']:.4f}")
print(f"Token Accuracy: {metrics['token_accuracy']:.4f}")
print(f"TP: {metrics['true_positive']}")
print(f"FP: {metrics['false_positive']}")
print(f"FN: {metrics['false_negative']}")

df = pd.DataFrame(rows)

df.to_csv("Claude_Opus_4_7_FLARE_CD_Evaluation_Results.csv", index=False)

with open("Claude_Opus_4_7_FLARE_CD_Metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

num_errors = df["error"].astype(str).str.len().gt(0).sum()

print("\n===== Error Summary =====")
print(f"Failed samples: {num_errors}")
print(f"Total samples: {len(df)}")
print(f"Failure rate: {num_errors / len(df):.4f}")

pd.set_option("display.max_colwidth", 120)
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/519 [00:00<?, ?B/s]

data/test-00000-of-00001-29751a6cc0ca838(…):   0%|          | 0.00/195k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/226 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['id', 'query', 'answer', 'text', 'label', 'token'],
        num_rows: 226
    })
})
Split: test
Columns: ['id', 'query', 'answer', 'text', 'label', 'token']
Example: {'id': 'cd0', 'query': "Your job in this task is to perform sequence labeling on a provided text section, marking the chunks that represent the cause of an event and the effects that result from it. For each token in the text, assign a label to indicate its role in representing cause or effect. The labels you should use are 'B-CAUSE', 'I-CAUSE', 'B-EFFECT', 'I-EFFECT', and 'O'. A 'B-' prefix is used to denote the beginning of a cause or effect sequence, while an 'I-' prefix is used for continuation of a cause or effect sequence. If a token is not part of either a cause or effect sequence, label it as 'O'. Provide your answer as a sequence of 'token:label' pairs, with each pair on a new line.\nText: Around 21,000 employees , 9,000 of whom are employed in the UK , are to b

100%|██████████| 100/100 [16:35<00:00,  9.95s/it]


===== Claude Opus 4.7 FLARE-CD Evaluation =====
Dataset: ChanceFocus/flare-cd
Split: test
Model: claude-opus-4-7
Samples evaluated: 100
Precision / Correctness Rate: 0.2673
Recall: 0.1350
F1 Score: 0.1794
Exact Match Accuracy: 0.0300
Token Accuracy: 0.3216
TP: 27
FP: 74
FN: 173

===== Error Summary =====
Failed samples: 0
Total samples: 100
Failure rate: 0.0000


,idx,id,text,tokens,gold_labels,pred_labels,gold_spans,pred_spans,exact_match,error
0,0,cd0,"Around 21,000 employees , 9,000 of whom are employed in the UK , are to be made redundant after the 178-year-old com...","[Around, 21,000, employees, ,, 9,000, of, whom, are, employed, in, the, UK, ,, are, to, be, made, redundant, after, ...","[B-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFF...","[B-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFF...","[(0, 18, EFFECT), (19, 31, CAUSE)]","[(0, 17, EFFECT), (18, 31, CAUSE)]",False,
1,1,cd1,REUTERS/Aly Song/File Photo ( Reuters ) - Tencent Holdings Ltd and private equity partner Hammer Capital have offere...,"[REUTERS/Aly, Song/File, Photo, (, Reuters, ), -, Tencent, Holdings, Ltd, and, private, equity, partner, Hammer, Cap...","[O, O, O, O, O, O, O, B-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CA...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O...","[(7, 36, CAUSE), (37, 46, EFFECT)]",[],False,
2,2,cd2,"Finally , Bank of America reduced their price target on Intrexon from $ 7.00 to $ 6.00 and set an underperform ratin...","[Finally, ,, Bank, of, America, reduced, their, price, target, on, Intrexon, from, $, 7.00, to, $, 6.00, and, set, a...","[B-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O...","[(0, 34, CAUSE), (64, 81, EFFECT)]",[],False,
3,3,cd3,"RWR traded up $ 0.29 during trading on Wednesday , hitting $ 103.68.","[RWR, traded, up, $, 0.29, during, trading, on, Wednesday, ,, hitting, $, 103.68.]","[B-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, I-CAUSE, O, B-EFFECT, I-EFFECT, I-EFFECT]","[O, O, O, O, O, O, O, O, O, O, O, O, O]","[(0, 9, CAUSE), (10, 13, EFFECT)]",[],False,
4,4,cd4,"More than 20,000 jobs across the group are at risk if it collapses , with 9,000 of those jobs in the UK.","[More, than, 20,000, jobs, across, the, group, are, at, risk, if, it, collapses, ,, with, 9,000, of, those, jobs, in...","[B-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, O, B-CAUSE, I-C...","[B-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, I-EFFECT, O, B-CAUSE, I-C...","[(0, 10, EFFECT), (11, 22, CAUSE)]","[(0, 10, EFFECT), (11, 13, CAUSE)]",False,
